# AFM Colab Environment Check

Use this notebook to verify the runtime, package requirements, GPU availability, and dataset paths before retraining.


## Setup

This version mounts Google Drive, lists the top of `MyDrive`, and recursively searches for the actual repo root automatically.


In [24]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
from pathlib import Path

mydrive = Path('/content/drive/MyDrive')
print('MyDrive exists:', mydrive.exists())
if mydrive.exists():
    top_entries = [child.name for child in sorted(mydrive.iterdir())[:100]]
    print('Top entries in MyDrive:')
    for name in top_entries:
        print(' -', name)


MyDrive exists: True
Top entries in MyDrive:
 - (Electronic) FYP1_FYP2 SV_Examiner 2024.xlsx
 - (Elektronik) FYP1_FYP2 SV_Examiner S2 SA 23-24.xlsx
 - (UL01) PERMOHONAN LATIHAN Dr Fairus (pdf.io).pdf
 - 045_Sakib_ICED2022.pdf
 - 1.0 KPI Form UniMAP Research Assessment - PPKME.gsheet
 - 1.0 KPI Form UniMAP Research Assessment - PPKME.xls
 - 10.1021@jacs.0c02835.pdf
 - 10Studio-Sample-EN.xlsx
 - 1912_Accepted (proof).docx
 - 1978_Accepted_proof.docx
 - 1989_Accepted_proof.docx
 - 1st MONITORING FORM (1).gform
 - 1st MONITORING FORM (Responses).gsheet
 - 1st MONITORING FORM 2026.gform
 - 1st MONITORING FORM.gform
 - 20 jan 23 DFT IRWANY updated.docx
 - 2021 FYP II Thesis Form (Examiner).pdf
 - 2021 FYP II Thesis Form (Supervisor) (1).pdf
 - 2021 FYP II Thesis Form (Supervisor).pdf
 - 2021 FYP II Thesis Form (Supervisor)NURUL SYAZWANI BINTI MOHAMAD TAUFIK.pdf
 - 2021 FYP II Viva.pdf
 - 2021 FYPEX Rubric Assessment Form (SITI NUR AFIFAH BINTI IDRIS) .pdf
 - 2021 FYPEX Rubric Assessment Form

In [26]:
import os
from pathlib import Path

MANUAL_REPO_DIR = ''  # optional override

def looks_like_repo(path: Path) -> bool:
    return (path / 'requirements.txt').exists() and (path / 'training').exists()

def find_repo_candidates(root: Path, limit=20):
    matches = []
    if not root.exists():
        return matches
    for req in root.rglob('requirements.txt'):
        candidate = req.parent
        if looks_like_repo(candidate):
            matches.append(candidate)
            if len(matches) >= limit:
                break
    return matches

candidate_roots = []
if MANUAL_REPO_DIR:
    candidate_roots.append(Path(MANUAL_REPO_DIR))
candidate_roots.extend([
    Path('/content/drive/MyDrive/AFM-Hysteresis-Simulation'),
    Path('/content/drive/MyDrive/Colab Notebooks/AFM-Hysteresis-Simulation'),
    Path('/content/AFM-Hysteresis-Simulation'),
    Path('/content/drive/MyDrive'),
    Path.cwd(),
])

repo = None
all_matches = []
seen = set()
for root in candidate_roots:
    root_key = str(root)
    if root_key in seen:
        continue
    seen.add(root_key)
    if root.exists() and looks_like_repo(root):
        repo = root
        break
    matches = find_repo_candidates(root)
    for match in matches:
        match_key = str(match)
        if match_key not in {str(m) for m in all_matches}:
            all_matches.append(match)
    if matches:
        repo = matches[0]
        break

if repo is None:
    print('Repo detection failed.')
    print('Checked roots:')
    for root in candidate_roots:
        print(' -', root, 'exists=', root.exists())
    print('\nRepo-like matches found:')
    if all_matches:
        for match in all_matches:
            print(' -', match)
    else:
        print(' - none')
    raise FileNotFoundError('Could not locate a repo root with requirements.txt and training/.')

os.chdir(repo)
print('REPO_DIR =', repo)
print('cwd =', Path.cwd())


REPO_DIR = /content/drive/MyDrive/AFM-Hysteresis-Simulation (1)
cwd = /content/drive/MyDrive/AFM-Hysteresis-Simulation (1)


In [27]:
!python --version
!nvidia-smi || true


Python 3.12.13
/bin/bash: line 1: nvidia-smi: command not found


In [28]:
!pip install -r requirements.txt


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 6.6 MB/s eta 0:00:00


In [29]:
import importlib

packages = [
    'numpy',
    'matplotlib',
    'cv2',
    'joblib',
    'pandas',
    'sklearn',
    'torch',
    'torchvision',
    'ultralytics',
]

for name in packages:
    try:
        module = importlib.import_module(name)
        version = getattr(module, '__version__', 'unknown')
        print(f'{name}: OK ({version})')
    except Exception as exc:
        print(f'{name}: FAIL ({exc})')


numpy: OK (2.0.2)
matplotlib: OK (3.10.0)
cv2: OK (5.0.0)
joblib: OK (1.5.3)
pandas: OK (2.2.2)
sklearn: OK (1.6.1)
torch: OK (2.11.0+cpu)
torchvision: OK (0.26.0+cpu)
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
ultralytics: OK (8.4.120)


In [30]:
import torch

print('torch.cuda.is_available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu_name =', torch.cuda.get_device_name(0))
    print('device_count =', torch.cuda.device_count())
else:
    print('No CUDA GPU detected. Training can still run on CPU.')


torch.cuda.is_available = False
No CUDA GPU detected. Training can still run on CPU.


In [31]:
from pathlib import Path

checks = [
    Path('requirements.txt'),
    Path('training/run_camera_pov_retraining.py'),
    Path('collected_data/site_memories'),
    Path('collected_data/prepared_training'),
]

for path in checks:
    print(f'{path}:', path.exists())


requirements.txt: True
training/run_camera_pov_retraining.py: True
collected_data/site_memories: True
collected_data/prepared_training: True


In [32]:
from pathlib import Path

site_root = Path('collected_data/site_memories')
metadata_files = sorted(site_root.rglob('metadata.json')) if site_root.exists() else []
print('site_memory_count =', len(metadata_files))
for item in metadata_files[:5]:
    print(item)


site_memory_count = 49
collected_data/site_memories/ChatGPT_Image_Jul_17__2026__02_31_38_PM/20260729_213748_unlabeled_site/metadata.json
collected_data/site_memories/ChatGPT_Image_Jul_17__2026__02_31_38_PM/20260729_214057_unlabeled_site/metadata.json
collected_data/site_memories/ChatGPT_Image_Jul_17__2026__02_31_38_PM/20260809_175132_Origin/metadata.json
collected_data/site_memories/ChatGPT_Image_Jul_17__2026__02_31_38_PM/20260809_205132_Origin/metadata.json
collected_data/site_memories/ChatGPT_Image_Jul_17__2026__02_31_38_PM/20260813_213232_unlabeled_site/metadata.json


In [33]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from training.prepare_camera_fov_dataset import inspect_site_memory, iter_site_memory_dirs

rows = [inspect_site_memory(site_dir) for site_dir in iter_site_memory_dirs(PROJECT_ROOT / 'collected_data/site_memories')]
camera_ready = sum(1 for row in rows if row['has_live_camera_view'])
legacy = len(rows) - camera_ready
print('total_site_memories =', len(rows))
print('camera_fov_ready =', camera_ready)
print('legacy =', legacy)


total_site_memories = 49
camera_fov_ready = 19
legacy = 30


## Pass criteria

- `requirements.txt` installs without error
- key imports report `OK`
- `collected_data/site_memories` exists
- at least some site memories are listed
- if using GPU, `torch.cuda.is_available()` is `True`
